In [5]:
# ==================================================
# 부품별 AI 학습 상태 검사
# ==================================================

import os

import torch

from torchvision import transforms
from PIL import Image

# ==================================================
# 1. 검사할 부품
# ==================================================

objects = [
    "nipper",
    "pen",
    "wire_stripper"
]


# ==================================================
# 2. 이미지 전처리
# ==================================================

test_transform = transforms.Compose([

    transforms.Resize(
        (128, 128)
    ),

    transforms.ToTensor()
])


# ==================================================
# 3. 모델 불러오기
# ==================================================

def load_test_model(target):

    model_file = (
        f"models/{target}_model.pth"
    )


    checkpoint = torch.load(
        model_file,
        map_location="cpu"
    )


    model = CNN()


    model.load_state_dict(
        checkpoint["model_state"]
    )


    model.eval()


    return model


# ==================================================
# 4. 특정 이미지 하나 테스트
# ==================================================

def predict_image(
    model,
    image_path
):

    image = Image.open(
        image_path
    ).convert("RGB")


    image = test_transform(
        image
    )


    image = image.unsqueeze(0)


    with torch.no_grad():

        output = model(
            image
        )


        probability = torch.softmax(
            output,
            dim=1
        )


    other_probability = (
        probability[0, 0].item()
    )


    target_probability = (
        probability[0, 1].item()
    )


    return (
        other_probability,
        target_probability
    )


# ==================================================
# 5. 모델 하나 검사
# ==================================================

def test_one_model(
    target
):

    print()

    print("================================")
    print(f"🎯 {target} 모델 검사")
    print("================================")


    # 모델 불러오기
    model = load_test_model(
        target
    )


    total_target = 0
    correct_target = 0


    total_other = 0
    correct_other = 0


    # --------------------------------------------------
    # 모든 부품 폴더 검사
    # --------------------------------------------------

    for object_name in objects:


        folder = os.path.join(
            "../dataset",
            object_name
        )


        files = [

            file

            for file in os.listdir(folder)

            if file.lower().endswith(
                (
                    ".jpg",
                    ".jpeg",
                    ".png"
                )
            )
        ]


        for file in files:

            image_path = os.path.join(
                folder,
                file
            )


            other_probability, target_probability = (
                predict_image(
                    model,
                    image_path
                )
            )


            # --------------------------------------------------
            # 실제 목표 부품
            # --------------------------------------------------

            if object_name == target:

                total_target += 1


                if target_probability >= 0.5:

                    correct_target += 1


            # --------------------------------------------------
            # 다른 부품
            # --------------------------------------------------

            else:

                total_other += 1


                if target_probability < 0.5:

                    correct_other += 1


    # --------------------------------------------------
    # 정확도 계산
    # --------------------------------------------------

    target_accuracy = (

        correct_target
        / total_target
        * 100

    )


    other_accuracy = (

        correct_other
        / total_other
        * 100

    )


    total_correct = (
        correct_target
        + correct_other
    )


    total_images = (
        total_target
        + total_other
    )


    total_accuracy = (

        total_correct
        / total_images
        * 100

    )


    # --------------------------------------------------
    # 결과 출력
    # --------------------------------------------------

    print()

    print(
        f"실제 {target}: "
        f"{correct_target}/{total_target} "
        f"→ {target_accuracy:.1f}%"
    )


    print(
        f"다른 부품: "
        f"{correct_other}/{total_other} "
        f"→ {other_accuracy:.1f}%"
    )


    print(
        f"전체 정확도: "
        f"{total_correct}/{total_images} "
        f"→ {total_accuracy:.1f}%"
    )


    return (
        target_accuracy,
        other_accuracy,
        total_accuracy
    )


# ==================================================
# 6. 모든 모델 검사
# ==================================================

results = {}


for target in objects:

    results[target] = test_one_model(
        target
    )


# ==================================================
# 7. 최종 결과
# ==================================================

print()

print("================================")
print("       📊 최종 검사 결과")
print("================================")

print()


for target, result in results.items():

    target_accuracy = result[0]

    other_accuracy = result[1]

    total_accuracy = result[2]


    print(
        f"{target}: "
        f"목표물 {target_accuracy:.1f}% / "
        f"다른 부품 {other_accuracy:.1f}% / "
        f"전체 {total_accuracy:.1f}%"
    )


🎯 nipper 모델 검사


NameError: name 'CNN' is not defined